# 04 — Client partitioning strategies

**Purpose.** How data is split across simulated clients: IID, Dirichlet, label shards, stratified, quantity skew; missing-data simulation.

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/04_client_partitioning_*.png|pdf` and `reports/notebooks/04_client_partitioning.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "04_client_partitioning"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Partition SYN-IDS-v1 ten ways

In [2]:
from federated_learning.synthetic_data import make_dataset
from federated_learning.data_partitioning import partition, label_skew_stat, simulate_missing
d = make_dataset(n_samples=6000, seed=SEED); y = d.y_train
cfgs = {"iid": {}, "stratified": {}, "dirichlet a=0.5": dict(alpha=.5), "dirichlet a=0.1": dict(alpha=.1), "label_shards": dict(shards_per_client=1), "quantity_skew": dict(sigma=1.5)}
P = {k: partition(k.split()[0], y, 10, SEED, **v) for k, v in cfgs.items()}
tab = pd.DataFrame([dict(strategy=k, min_size=min(map(len, p)), max_size=max(map(len, p)), label_skew=round(label_skew_stat(y, p), 3)) for k, p in P.items()]); display(tab); SUMMARY["partitions"] = tab.to_dict("records")

,strategy,min_size,max_size,label_skew
0,iid,420,420,0.019
1,stratified,419,421,0.000
2,dirichlet a=0.5,7,1535,0.317
3,dirichlet a=0.1,5,2424,0.477
4,label_shards,420,420,0.479
5,quantity_skew,36,1501,0.021


## 2. Figure: positive rate per client

In [3]:
fig, ax = plt.subplots(2, 3, figsize=(7.4, 3.6), sharey=True)
for a, (k, p) in zip(ax.ravel(), P.items()):
    a.bar(range(10), [y[i].mean() for i in p], color=PAL[0]); a.axhline(y.mean(), color=PAL[1], lw=1); a.set_title(k, fontsize=8)
fig.supxlabel("client"); fig.supylabel("positive rate"); save_fig(fig, "partitions")

## 3. Missing-data simulation (MCAR / MAR / MNAR)

In [4]:
X = d.X_train[:2000]
for m in ("mcar", "mar", "mnar"): print(m, "missing fraction:", round(float(np.isnan(simulate_missing(X, .2, m, SEED)).mean()), 3))
save_summary()

mcar missing fraction: 0.2
mar missing fraction: 0.195
mnar missing fraction: 0.199
summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/04_client_partitioning.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.